Deep Dive: Regularization Terminology and Concepts
L2 regularization adds a penalty term to the cost function proportional to the square of the weight magnitudes

Dropout
What it is:
A regularization technique that randomly "drops out" (sets to zero) a proportion of neurons during each training iteration.

Inverted Dropout
What it is:
A scaling technique that maintains the expected value of activations during training.

1. NumPy - import numpy as np
Purpose: Numerical computing, array operations

Used for: Matrix multiplications, random number generation, mathematical operations

2. Matplotlib - import matplotlib.pyplot as plt
Purpose: Data visualization

Used for: Plotting cost curves, decision boundaries, dataset visualization

3. Scikit-learn - import sklearn, sklearn.datasets
Purpose: Machine learning utilities

Used for: Dataset loading, preprocessing

4. SciPy - import scipy.io
Purpose: Scientific computing

Used for: Loading .mat files (MATLAB data format)

5. reg_utils.py (Custom module)
This file contains:

sigmoid() - Sigmoid activation function

relu() - ReLU activation function

plot_decision_boundary() - Visualization helper

initialize_parameters() - Weight initialization

load_2D_dataset() - Load the football dataset

predict_dec() - Prediction for decision boundary

compute_cost() - Cross-entropy cost

predict() - Make predictions

forward_propagation() - Forward pass without regularization

backward_propagation() - Backward pass without regularization

update_parameters() - Gradient descent update

6. testCases.py
Purpose: Provides test cases for grading

Contains: Pre-computed values for function testing

7. public_tests.py
Purpose: Automated grading tests

Contains: Test functions for each exercise

8. IPython utilities
Purpose: Enhanced notebook experience

Used for: Displaying videos, images, and interactive content

In [12]:
# Regularization Assignment - Complete Imports

# Core scientific computing
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import sklearn.datasets
import scipy.io

# Deep learning specific functions from assignment files
from reg_utils import (
    sigmoid, 
    relu, 
    plot_decision_boundary, 
    initialize_parameters, 
    load_2D_dataset, 
    predict_dec,
    compute_cost, 
    predict, 
    forward_propagation, 
    backward_propagation, 
    update_parameters
)

# Test cases and public tests for grading
from testCases import *
from public_tests import *

# IPython utilities for better notebook experience
import IPython
from IPython.display import display, HTML, Image, Video

# Magic commands for matplotlib inline display
%matplotlib inline

# Set default plot parameters
plt.rcParams['figure.figsize'] = (7.0, 4.0)  # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

# Optional: Set random seeds for reproducibility
np.random.seed(3)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
def compute_cost_with_regularization(A3, Y, parameters, lambd):
    """
    Implement the cost function with L2 regularization.
    """
    m = Y.shape[1]
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    W3 = parameters["W3"]
    
    cross_entropy_cost = compute_cost(A3, Y)
    
    # L2 regularization term
    L2_regularization_cost = (lambd/(2*m)) * (np.sum(np.square(W1)) + np.sum(np.square(W2)) + np.sum(np.square(W3)))
    
    cost = cross_entropy_cost + L2_regularization_cost
    
    return cost

Exercise 2 - backward_propagation_with_regularization


In [14]:
def backward_propagation_with_regularization(X, Y, cache, lambd):
    """
    Implements backward propagation with L2 regularization.
    """
    m = X.shape[1]
    (Z1, A1, W1, b1, Z2, A2, W2, b2, Z3, A3, W3, b3) = cache
    
    dZ3 = A3 - Y
    dW3 = 1./m * np.dot(dZ3, A2.T) + (lambd/m) * W3
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)
    
    dA2 = np.dot(W3.T, dZ3)
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1./m * np.dot(dZ2, A1.T) + (lambd/m) * W2
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1./m * np.dot(dZ1, X.T) + (lambd/m) * W1
    db1 = 1. / m * np.sum(dZ1, axis=1, keepdims=True)
    
    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1, 
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}
    
    return gradients

Exercise 3 - forward_propagation_with_dropout


In [15]:
def forward_propagation_with_dropout(X, parameters, keep_prob=0.5):
    """
    Implements forward propagation with dropout.
    """
    np.random.seed(1)
    
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]
    
    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    
    # Step 1: Initialize D1 matrix with random values
    D1 = np.random.rand(A1.shape[0], A1.shape[1])
    # Step 2: Convert to 0/1 based on keep_prob threshold
    D1 = (D1 < keep_prob).astype(int)
    # Step 3: Shut down some neurons
    A1 = A1 * D1
    # Step 4: Scale the value
    A1 = A1 / keep_prob
    
    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)
    
    # Step 1: Initialize D2 matrix with random values
    D2 = np.random.rand(A2.shape[0], A2.shape[1])
    # Step 2: Convert to 0/1 based on keep_prob threshold
    D2 = (D2 < keep_prob).astype(int)
    # Step 3: Shut down some neurons
    A2 = A2 * D2
    # Step 4: Scale the value
    A2 = A2 / keep_prob
    
    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)
    
    cache = (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)
    
    return A3, cache

Exercise 4 - backward_propagation_with_dropout


In [16]:
def backward_propagation_with_dropout(X, Y, cache, keep_prob):
    """
    Implements backward propagation with dropout.
    """
    m = X.shape[1]
    (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3) = cache
    
    dZ3 = A3 - Y
    dW3 = 1./m * np.dot(dZ3, A2.T)
    db3 = 1./m * np.sum(dZ3, axis=1, keepdims=True)
    dA2 = np.dot(W3.T, dZ3)
    
    # Step 1: Apply mask D2
    dA2 = dA2 * D2
    # Step 2: Scale the value
    dA2 = dA2 / keep_prob
    
    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1./m * np.dot(dZ2, A1.T)
    db2 = 1./m * np.sum(dZ2, axis=1, keepdims=True)
    
    dA1 = np.dot(W2.T, dZ2)
    
    # Step 1: Apply mask D1
    dA1 = dA1 * D1
    # Step 2: Scale the value
    dA1 = dA1 / keep_prob
    
    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1./m * np.dot(dZ1, X.T)
    db1 = 1./m * np.sum(dZ1, axis=1, keepdims=True)
    
    gradients = {"dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
                 "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1, 
                 "dZ1": dZ1, "dW1": dW1, "db1": db1}
    
    return gradients

Why Regularization Hurts Training Accuracy
Overfitting Analogy (Exam Studying):

No Regularization: Student memorizes entire textbook (including typos)

Perfect on practice problems (training)

Fails when questions are rephrased (test)

With Regularization: Student learns core concepts

Might miss some details in practice (lower training)

Can solve new variations of problems (higher test)

Regularization techniques are like training wheels for neural networks:

They constrain the learning process

Prevent memorization of noise

Force learning of generalizable patterns

The slight decrease in training performance is a small price for much better real-world performance!